# Experiment 5: MLlib Algorithms using Apache Spark (PySpark)

## Aim
To implement Machine Learning algorithms such as Classification (Logistic Regression) and Clustering (K-Means) using Apache Spark MLlib.

## Step 1: Install PySpark
Since Colab runs on Linux, we can simply install pyspark and use it without complex configuration.

In [ ]:
!pip install pyspark

## Step 2: Create Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import Row
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Create Spark Session
spark = SparkSession.builder \
    .appName("MLlib_Lab_Colab") \
    .getOrCreate()

# No special Windows config needed here!
print("Spark Session Created Successfully!")

--- 
## Experiment - A: Solution with In-Code Data

### 1. Classification (Logistic Regression)

In [ ]:
# Create Sample Data
data = [
    Row(age=22, salary=20000, label=0),
    Row(age=25, salary=25000, label=0),
    Row(age=35, salary=50000, label=1),
    Row(age=45, salary=80000, label=1)
]

df = spark.createDataFrame(data)
print("Raw Data:")
df.show()

# Feature Vectorization
assembler = VectorAssembler(
    inputCols=["age", "salary"],
    outputCol="features"
)

final_data = assembler.transform(df)
final_data.show()

# Model Training
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(final_data)

# Prediction
predictions = model.transform(final_data)
print("Logistic Regression Predictions:")
predictions.select("age", "salary", "label", "prediction").show()

### 2. Clustering (K-Means)

In [ ]:
# Create Sample Dataset
data_kmeans = [
    (1.0, 1.0),
    (1.5, 2.0),
    (3.0, 4.0),
    (5.0, 7.0),
    (3.5, 5.0),
    (4.5, 5.0),
    (3.5, 4.5)
]

columns = ["x", "y"]
df_kmeans = spark.createDataFrame(data_kmeans, columns)
print("K-Means Data:")
df_kmeans.show()

# Convert to Feature Vector
assembler_kmeans = VectorAssembler(
    inputCols=["x", "y"],
    outputCol="features"
)

final_df_kmeans = assembler_kmeans.transform(df_kmeans)

# Apply K-Means
kmeans = KMeans(k=2, seed=1)
model_kmeans = kmeans.fit(final_df_kmeans)

# Predict Clusters
predictions_kmeans = model_kmeans.transform(final_df_kmeans)
print("K-Means Predictions:")
predictions_kmeans.select("x", "y", "prediction").show()

# Cluster Centers
centers = model_kmeans.clusterCenters()
print("Cluster Centers:")
for center in centers:
    print(center)

--- 
## Experiment - B: Solution with CSV Data
We will first create the CSV files programmatically so this notebook is self-contained.

In [ ]:
# Create data.csv
with open("data.csv", "w") as f:
    f.write("age,salary,label\n")
    f.write("22,20000,0\n")
    f.write("25,25000,0\n")
    f.write("35,50000,1\n")
    f.write("45,80000,1\n")

# Create cluster_data.csv
with open("cluster_data.csv", "w") as f:
    f.write("x,y\n")
    f.write("1.0,1.0\n")
    f.write("1.5,2.0\n")
    f.write("3.0,4.0\n")
    f.write("5.0,7.0\n")
    f.write("3.5,5.0\n")
    f.write("4.5,5.0\n")
    f.write("3.5,4.5\n")

print("CSV files created successfully.")

In [ ]:
# Step 1: Load Dataset
data_csv = spark.read.csv("data.csv", header=True, inferSchema=True)

# Step 2: Feature Vectorization
assembler_csv = VectorAssembler(
    inputCols=["age", "salary"],
    outputCol="features"
)

final_data_csv = assembler_csv.transform(data_csv)
final_data_csv = final_data_csv.select("features", "label")

# Step 3: Split Dataset
train_data, test_data = final_data_csv.randomSplit([0.7, 0.3], seed=42)

# Step 4: Train Logistic Regression Model
lr_csv = LogisticRegression(featuresCol="features", labelCol="label")
model_csv = lr_csv.fit(train_data)

# Step 5: Predictions
predictions_csv = model_csv.transform(test_data)
print("Predictions on Test Data:")
predictions_csv.select("label", "prediction").show()

# Step 6: Model Evaluation
evaluator_csv = BinaryClassificationEvaluator()
accuracy_csv = evaluator_csv.evaluate(predictions_csv)
print("Accuracy:", accuracy_csv)

In [ ]:
# Clustering with CSV Data
data_cluster_csv = spark.read.csv("cluster_data.csv", header=True, inferSchema=True)

assembler_cluster_csv = VectorAssembler(
    inputCols=["x", "y"],
    outputCol="features"
)

final_data_cluster_csv = assembler_cluster_csv.transform(data_cluster_csv)

kmeans_csv = KMeans(k=3, seed=1)
model_kmeans_csv = kmeans_csv.fit(final_data_cluster_csv)

centers_csv = model_kmeans_csv.clusterCenters()
print("Cluster Centers (from CSV):")
for center in centers_csv:
    print(center)